# Docker Basics for AI Engineers

This notebook covers Docker fundamentals needed to containerize and deploy AI applications. All Docker commands run in your terminal — this notebook provides the theory, reference, and examples.

## 1. Why Docker for AI?

AI applications have unique packaging challenges:

| Challenge | Without Docker | With Docker |
|-----------|---------------|-------------|
| "Works on my machine" | Dependency conflicts, OS differences | Identical environment everywhere |
| Large ML libraries | Manual install, version conflicts | Pre-built layers, cached downloads |
| Model artifacts | Copy files manually | Bake into image or mount as volume |
| Deployment | Manual server setup | One command deploy |
| Scaling | Complex orchestration | Cloud Run / Kubernetes handle it |

**Docker solves the packaging problem.** You build an image once, run it anywhere.

## 2. Docker Concepts

### Image vs Container

```
Image (Blueprint)          Container (Running Instance)
┌──────────────┐           ┌──────────────┐
│  Python 3.11  │    ──▶    │  Running app  │
│  FastAPI      │  docker   │  Serving on   │
│  Your code    │   run     │  port 8000    │
│  Dependencies │           │  Processing   │
└──────────────┘           │  requests     │
                           └──────────────┘
   Read-only                Stateful, ephemeral
```

- **Image**: Immutable template. Created from a Dockerfile. Stored in registries.
- **Container**: Running process created from an image. Ephemeral (destroyed when stopped).
- **Dockerfile**: Build script — set of instructions to create an image.
- **Registry**: Image storage (Docker Hub, GCR, ECR).

## 3. Dockerfile Anatomy

Here's a Dockerfile for a Python/FastAPI app with explanations:

```dockerfile
# ---- Stage 1: Build ----
FROM python:3.11-slim AS builder

WORKDIR /app

# Install dependencies first (cached layer)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ---- Stage 2: Runtime ----
FROM python:3.11-slim

WORKDIR /app

# Copy only installed packages from builder
COPY --from=builder /usr/local/lib/python3.11/site-packages /usr/local/lib/python3.11/site-packages
COPY --from=builder /usr/local/bin /usr/local/bin

# Copy application code
COPY . .

EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Layer Caching

Docker builds images in layers. Each instruction creates a new layer:

```
Layer 1: FROM python:3.11-slim      (base OS + Python)
Layer 2: COPY requirements.txt     (small file)
Layer 3: RUN pip install ...       (slow, but cached if requirements unchanged)
Layer 4: COPY . .                  (your code — changes often)
```

**Key insight**: Put `requirements.txt` before `COPY . .` so dependency installation is cached when only code changes.

## 4. Docker Commands Reference

Run these in your terminal (not in the notebook).

### Build & Run

```bash
# Build an image
docker build -t ai-app:latest .

# Build with a specific Dockerfile
docker build -t ai-app:latest -f Dockerfile.prod .

# Run a container
docker run -p 8000:8000 ai-app:latest

# Run in detached mode
docker run -d -p 8000:8000 --name my-ai-app ai-app:latest

# Run with environment variables
docker run -p 8000:8000 -e OPENAI_API_KEY=xxx ai-app:latest

# Run with .env file
docker run -p 8000:8000 --env-file .env ai-app:latest
```

### Manage Containers

```bash
# List running containers
docker ps

# List all containers (including stopped)
docker ps -a

# Stop a container
docker stop my-ai-app

# Remove a container
docker rm my-ai-app

# View logs
docker logs my-ai-app
docker logs -f my-ai-app  # follow (like tail -f)

# Execute command in running container
docker exec -it my-ai-app /bin/bash
```

### Images

```bash
# List local images
docker images

# Remove an image
docker rmi ai-app:latest

# Clean up unused images/containers
docker system prune -a

# Inspect image layers
docker history ai-app:latest
```

### Registry

```bash
# Tag for a registry
docker tag ai-app:latest gcr.io/my-project/ai-app:latest

# Push to registry
docker push gcr.io/my-project/ai-app:latest

# Pull from registry
docker pull gcr.io/my-project/ai-app:latest
```

## 5. Common Docker Patterns for AI Apps

### Pattern 1: Multi-stage Build (Smaller Images)

```dockerfile
# Build stage — full Python image with build tools
FROM python:3.11 AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

# Runtime stage — minimal image
FROM python:3.11-slim
WORKDIR /app
COPY --from=builder /install /usr/local
COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Pattern 2: Separate Requirements from Code

```dockerfile
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
# Code changes won't reinstall dependencies
COPY . .
```

### Pattern 3: Non-Root User

```dockerfile
FROM python:3.11-slim
RUN useradd --create-home appuser
WORKDIR /app
COPY --chown=appuser:appuser . .
USER appuser
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Pattern 4: .dockerignore

Create a `.dockerignore` file to exclude files from the build context:

```gitignore
.git
.gitignore
.env
.env.local
__pycache__
*.pyc
.venv
venv
node_modules
.pytest_cache
*.ipynb
notebooks/
exercises/
README.md
```

## 6. Verifying Your Docker Setup

Run these commands in your terminal to verify Docker is installed and working:

```bash
# Check Docker version
docker --version
# Expected: Docker version 24.x or 25.x

# Check Docker Compose version
docker compose version
# Expected: Docker Compose version v2.x

# Run the hello-world container
docker run hello-world
# If this works, Docker is properly installed

# Check available disk space (Docker images can be large)
docker system df
```

**If Docker Desktop is not installed**: Download from https://docs.docker.com/get-docker/

## 7. Quick Practice

### Exercise: Build and Run a Simple Python Container

1. Create a file called `hello.py`:

```python
from fastapi import FastAPI
app = FastAPI()

@app.get("/")
async def root():
    return {"message": "Hello from Docker!"}

@app.get("/health")
async def health():
    return {"status": "healthy"}
```

2. Create `requirements.txt`:

```
fastapi==0.115.0
uvicorn[standard]==0.30.0
```

3. Create `Dockerfile`:

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "hello:app", "--host", "0.0.0.0", "--port", "8000"]
```

4. Build and run:

```bash
docker build -t hello-app .
docker run -p 8000:8000 hello-app
```

5. Visit http://localhost:8000 in your browser.

6. Stop the container with `Ctrl+C` or `docker stop <container-id>`.

## Summary

| Concept | What It Is | Command |
|---------|-----------|---------|
| Image | Read-only template | `docker build -t name .` |
| Container | Running instance | `docker run -p 8000:8000 name` |
| Dockerfile | Build instructions | Edit and `docker build` |
| Registry | Image storage | `docker push` / `docker pull` |
| Layer caching | Speed up rebuilds | Order Dockerfile layers carefully |

**Next**: [02-containerize-ai-app.ipynb](02-containerize-ai-app.ipynb) — Containerize a complete FastAPI + AI backend.